# Q04：反复检索是在纠错，还是把正确结果改错？

状态：`implemented / numerical-validation-pending`。本文件未保存执行输出，不能预先回答哪个模型更好。

同一批记忆、同一批受损查询，比较 Classical Hopfield、Polynomial DAM 和连续迭代 Modern Hopfield。主图横轴是更新轮数，纵轴是目标身份识别率，每条线一个模型。t=0 是“不更新，直接识别查询”的共同对照。

**使用：保存一份到 Drive → 从头运行全部单元 → 查看验收结果和三模型曲线。** 默认 CPU、小规模 432 次检索。需要授权挂载 Drive 才能跨运行时恢复；Notebook 会自动加载多个源码文件，你只操作本文件。

继续沿用 `a 记忆 → b 存储 → c 查询 → d 更新 → e 测量 → f 同图比较`。模型代码在 `src/am_bench/models/`，配置与实验解释留在这里。[全部 13 个问题](https://github.com/Heptazero/nn-labs/blob/main/associative-memory/benchmarks/am-bench/QUESTIONS.md)。

## 0. 环境与固定代码版本

不需要额外安装依赖。下面下载公开仓库的固定提交，并记录实际代码摘要与依赖版本。修改 Notebook 的任务配置会进入运行记录；修改运行器或模型后应提交为新代码版本，更新 CODE_REV，并使用新结果目录。不要在同一个缓存目录直接改源码后继续基线运行。

In [ ]:
# [环境] Colab 预装依赖；自动获取固定版本的共享代码。
# 只需打开本 Notebook，不需要手动上传多个 Python 文件。
from pathlib import Path
import subprocess
import sys
import tempfile

CODE_REV = "b3d54f01a9cc890717abaab47af2b3f0768a82b6"
cache_root = Path("/content") if Path("/content").exists() else Path(tempfile.gettempdir())
REPO = cache_root / ("nn-labs-" + CODE_REV[:12])
if not REPO.exists():
    subprocess.run(["git", "init", str(REPO)], check=True, capture_output=True)
    subprocess.run(["git", "remote", "add", "origin", "https://github.com/Heptazero/nn-labs.git"],
                   cwd=REPO, check=True, capture_output=True)
# 未完成的下载可重试；已有不同 checkout 则拒绝覆盖。
head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True)
if head.returncode:
    subprocess.run(["git", "fetch", "--depth=1", "origin", CODE_REV], cwd=REPO, check=True)
    subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO, check=True)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
if actual != CODE_REV:
    raise RuntimeError("缓存 checkout 版本不符；请保留改动并使用新运行时。")
if subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip():
    raise RuntimeError("共享代码已有修改；请另存修改并使用干净运行时完成基线验收。")
if "am_bench" in sys.modules and Path(sys.modules["am_bench"].__file__).resolve().parent != REPO / "am_bench":
    raise RuntimeError("已载入其他版本的 am_bench；请重启会话后运行。")
sys.path.insert(0, str(REPO))
import am_bench
assert am_bench.API_VERSION == 1
from am_bench.provenance import source_info
SOURCE = source_info()
assert SOURCE["git_commit"] == CODE_REV
print("Loaded shared code:", SOURCE)

from dataclasses import asdict
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from am_bench.dynamics import DynamicsConfig, run_dynamics, transition_table, plot_dynamics_curves
plt.rcParams["figure.dpi"] = 120

## 1. 先验证拆分没有改变旧模型

对冻结旧版的六个模型检查 36 个小样本：终态、停止状态、步数、能量和误差轨迹。离散终态要求完全相同；连续终态和轨迹使用 `rtol=atol=1e-12`。另外检查观察器不能修改模型状态、外部目标不影响更新、连续输出确实反馈到下一轮，以及逐批恢复不会重复计算已有结果。

这些检查在执行本单元后才有结果；本地静态检查不能代替它们。失败时先修实现，不跳过检查去解释曲线。通过只表明小样本行为保持，不等于各论文已完整复现。

In [ ]:
# [验收] 对比冻结的拆分前版本；下载内容先核对 SHA-256，再执行指定模型定义。
NUMERICAL_GATE_PASSED = False
from urllib.request import urlopen
import hashlib
from am_bench.validation import verify_extraction, verify_dynamics

REFERENCE_REV = "e9adf216c5b5a0b329a102c0c25954e6ff48aa12"
reference = cache_root / ("nn-labs-reference-" + REFERENCE_REV[:12] + ".ipynb")
reference_url = ("https://raw.githubusercontent.com/Heptazero/nn-labs/" + REFERENCE_REV
                 + "/hopfield-benchmark/hopfield_benchmark_phase1_colab.ipynb")
if not reference.exists():
    with urlopen(reference_url, timeout=60) as response:
        reference.write_bytes(response.read())
if hashlib.sha256(reference.read_bytes()).hexdigest() != "7568d0bfe28703f24026dd49fc2756ecb7f58141ae2484e29652abace6d85be0":
    raise RuntimeError("旧版参考文件摘要不符；请删除该缓存文件后重新下载。")
print(verify_extraction(reference))
print(verify_dynamics())
NUMERICAL_GATE_PASSED = True

## 2. a / b / c / d：冻结这次要比较的任务

独立随机 ±1 记忆，每条查询翻转 `round(rho*N)` 位；所有模型读取完全相同的查询。默认三个模型各一个配置：经典模型去对角 Hebb、异步更新；Polynomial DAM 阶数 3、异步更新；Modern 固定 beta=0.1，把连续输出原样反馈，不重新取符号。

异步模型的一轮是逐个更新 N 个坐标；Modern 的一轮是一次完整连续读出。**轮数相同不等于计算成本相同**，此处回答原生更新过程的问题。beta=0.1 是预先选定的探索配置，尚未调优；不能由这张图宣称模型族排名或几何机制。

已登记的额外迭代模型有 `exponential_dam`、`simplicial_r12_t50`，可添加到 models；后者是异步扩展。PSHN 单步版本、FY 和双曲模型暂不参加这张逐轮主图。

In [ ]:
CONFIG = DynamicsConfig(
    N_values=(32, 64),
    P_values=(4, 8),
    corruption_levels=(0.0, 0.1, 0.3),
    seeds=(0, 1, 2),
    targets_per_set=4,
    max_sweeps=8,
    models=("classical_hebb", "polynomial_dam_d3", "continuous_modern_iterative"),
    modern_beta=0.1,
)
CONFIG.validate()
display(pd.Series(asdict(CONFIG), name="value").to_frame())
print(f"{CONFIG.retrieval_count} retrievals, {CONFIG.batch_count} batches; CPU float64")
print(f"{CONFIG.retrieval_count * (CONFIG.max_sweeps + 1)} rows including t=0 and explicit missing/held rows")

## 3. 分批保存到 Drive

一个 N/P/记忆库种子为一批，批内保留所有模型，完成后一次写入 JSON。每批带配置和内容摘要。中断时最多重算未保存的一批；同一目录重跑会读取已完成批次。运行结束仍建议检查 Drive 中的文件，挂载文件系统不等于事务数据库。

`RUN_NAME` 相同表示恢复同一次冻结实验；换配置、代码、Python 或 PyTorch 版本须换名字。一个目录只由一个运行时写入。关闭 Drive 时结果在临时运行时目录，不保证断线后还在。

`MAX_NEW_BATCHES=1` 可先做计时；默认 `None` 跑完小扫描。部分批次未完成时不会生成正式汇总图。

In [ ]:
RUN_NAME = "q04-smoke-v1"
MAX_NEW_BATCHES = None
if not RUN_NAME or any(c not in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-" for c in RUN_NAME):
    raise ValueError("RUN_NAME 请只用字母、数字、下划线或连字符")
OUTPUT = Path("/content/nn-labs-results") / RUN_NAME
print("Results directory:", OUTPUT)

## 4. e：逐轮测量并运行

正确答案只进入独立测量器，不进入模型更新或停止条件。共同 Top-1 使用“输出与哪条存储记忆内积最大”；并列取第一个索引，并保存并列数。它衡量身份识别，不能代替精确恢复。

每轮还保存均方误差、符号错误率、原生能量（如有）、权重熵/最大权重（如有）、停止状态和近似计算量。熵描述产生本轮输出的更新权重，不是校准后的可信概率。纯零查询或重复记忆可能产生并列，查看 tie_count 后再解释身份变化。

固定点之后的展示明确记为 held；数值失败后的缺测保留在主图分母，记为未成功，同时另画可用数据比例。已真实观测的失败前状态仍保留。达到步数上限只是 max_steps，不伪装成收敛。

In [ ]:
assert NUMERICAL_GATE_PASSED, "请先通过数值验收"
trajectories, progress = run_dynamics(CONFIG, OUTPUT, SOURCE, max_new_batches=MAX_NEW_BATCHES)
print(progress)
if not trajectories.empty:
    display(trajectories.head())
    last = trajectories[trajectories.step == CONFIG.max_sweeps]
    display(last.groupby(["model_id", "status"]).size().rename("trials").to_frame())
    print("Numerical failures:", int((last.status == "numerical_failure").sum()))
    print("Top-1 ties among available observations:", int((trajectories.top1_tie_count > 1).sum()))
if not progress["complete"]:
    print("尚未完成全部配对批次；重跑本单元继续。设置 MAX_NEW_BATCHES=None 可完成剩余批次。")

## 5. f：相同条件下一张图，每条线一个模型

左图使用全部配对查询；右图显示真实观测或明确保持的比例。阴影是三个记忆库之间的 ±1 标准误，小扫描只作探索，不能当作显著性检验。t=0 应对齐；若所有模型都早早达到 100%，说明当前任务太容易，不代表机制相同。

In [ ]:
SELECT_N = max(CONFIG.N_values)
SELECT_P = max(CONFIG.P_values)
SELECT_NOISE = max(CONFIG.corruption_levels)
if progress["complete"]:
    figure = plot_dynamics_curves(trajectories, N=SELECT_N, P=SELECT_P, corruption_level=SELECT_NOISE)
    figure.savefig(OUTPUT / "q04_success_and_coverage.png", bbox_inches="tight")
    plt.show()
else:
    print("等待完整扫描后作图。")

## 6. 继续一步，究竟救回了谁、又损坏了谁？

同时统计相邻轮和相对第一轮的变化。纠错率的分母是先前错误且两轮均可测的样本；改错率的分母是先前正确且两轮均可测的样本。分母为零记 N/A，不画成零。不同模型的“先前错了”人群不一定相同，因此条件率用于解释过程，跨模型主比较仍看上面的共同样本曲线。

In [ ]:
if progress["complete"]:
    adjacent = transition_table(trajectories)
    relative = transition_table(trajectories, relative_to_first=True)
    for label, table in (("adjacent", adjacent), ("relative_to_step1", relative)):
        table.to_csv(OUTPUT / (label + "_transitions.csv"), index=False)
        selected = table[(table.N == SELECT_N) & (table.P == SELECT_P)
                         & (table.corruption_level == SELECT_NOISE)]
        print(label)
        display(selected[["model_id", "step", "previous_wrong", "wrong_to_right", "correction_rate",
                          "previous_right", "right_to_wrong", "damage_rate"]])
    trajectories.to_csv(OUTPUT / "trajectory_summary.csv", index=False)
else:
    print("等待完整扫描后计算转移率。")

## 7. 成本与下一步判断

下面只报告各原生轮数下的平均累计近似 FLOPs（浮点运算次数估计），用于暴露开销差异。它不是硬件计数，不含建库/观测/排序成本，也没有完成同成本停止策略；不能据此宣称等算力排名。

In [ ]:
if progress["complete"]:
    selected = trajectories[(trajectories.N == SELECT_N) & (trajectories.P == SELECT_P)
                            & (trajectories.corruption_level == SELECT_NOISE)]
    cost = selected.groupby(["model_id", "step"], as_index=False).agg(
        top1_success=("top1_correct", "mean"),
        mean_estimated_flops=("estimated_cumulative_flops", "mean"),
        available_fraction=("available", "mean"),
    )
    display(cost)
    cost.to_csv(OUTPUT / "native_cost_summary.csv", index=False)
    print("原始批次、配置与派生图表：", OUTPUT)

暂时不要从“某条线更高”直接写新机制。先确认差异能否换种子复现，再分别改变负载、噪声和温度，检验哪个解释预言了变化。要研究“错误越来越确信”，下一步应在同一条错误轨迹上结合身份变化与权重集中度；本轮只收集这些字段，尚未做 Q05 的正式分析。

这份小扫描还未包含相关记忆、双曲模型、等计算预算停止、调参/测试分离和正式统计检验。先得到可复查的运行结果，再从 [问题目录](https://github.com/Heptazero/nn-labs/blob/main/associative-memory/benchmarks/am-bench/QUESTIONS.md) 选择下一项。

**验收标记：** 默认配置应显示 36 个拆分对照通过、dynamics/resume 通过、12/12 批次完成、3888 行轨迹、三条主曲线。另检查数值失败是否为零；有失败时保存 failure_reason 并排查。重启运行时后使用同一配置和目录，结果应直接从批次恢复。没有实际出现这些输出之前，状态仍是“待 Colab 验证”。